In [ ]:
# 1. Download Data
# tr_path = "covid.train.csv"
# tt_path = "covid.test.csv"

# 本地
tr_path = r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.train.csv"
tt_path = (
    r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.test.shuffle.csv"
)

# !gdown --id '19CCyCgJrUxtvgZF53vnctJiOJ23T5mqF' --output covid.train.csv
# !gdown --id '1CE240jLm2npU-tdz81-oVKEF3T2yfT1O' --output covid.test.csv

In [19]:
!pip install torch torchvision torchaudio

In [20]:
!pip install matplotlib scikit-learn optuna pyarrow openpyxl

In [ ]:
# 2. Import Packages
# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# For data preprocess
import numpy as np
import csv
import os

# For plotting
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure


def set_seed(seed=42069):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# 3. Some Utilities
def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def plot_learning_curve(loss_record, title=""):
    """Plot learning curve of your DNN (train & dev loss)"""
    total_steps = len(loss_record["train"])
    x_1 = range(total_steps)
    x_2 = x_1[:: len(loss_record["train"]) // len(loss_record["dev"])]
    figure(figsize=(6, 4))
    plt.plot(x_1, loss_record["train"], c="tab:red", label="train")
    plt.plot(x_2, loss_record["dev"], c="tab:cyan", label="dev")
    plt.ylim(0.0, 5.0)
    plt.xlabel("Training steps")
    plt.ylabel("MSE loss")
    plt.title("Learning curve of {}".format(title))
    plt.legend()
    plt.show()


def plot_pred(dv_set, model, device, lim=35.0, preds=None, targets=None):
    """Plot prediction of your DNN"""
    if preds is None or targets is None:
        model.eval()
        preds, targets = [], []
        for x, y in dv_set:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                preds.append(pred.detach().cpu())
                targets.append(y.detach().cpu())
        preds = torch.cat(preds, dim=0).numpy()
        targets = torch.cat(targets, dim=0).numpy()

    figure(figsize=(5, 5))
    plt.scatter(targets, preds, c="r", alpha=0.5)
    plt.plot([-0.2, lim], [-0.2, lim], c="b")
    plt.xlim(-0.2, lim)
    plt.ylim(-0.2, lim)
    plt.xlabel("ground truth value")
    plt.ylabel("predicted value")
    plt.title("Ground Truth v.s. Prediction")
    plt.show()


def save_pred(preds, file):
    """Save predictions to specified file"""
    print("Saving results to {}".format(file))
    with open(file, "w") as fp:
        writer = csv.writer(fp)
        writer.writerow(["id", "tested_positive"])
        for i, p in enumerate(preds):
            writer.writerow([i, p])


device = get_device()

In [ ]:
# 4. 主程式初始化與配置
config = {
    "train_path": r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.train.csv",
    "test_path": r"C:\Users\User\Desktop\Miracle\Master\上\DL\HW\HW1\data\covid.test.shuffle.csv",
    "save_dir": "models",
    "save_path": "models/best_model.pth",
    "seed": 42069,
    "target_only": True,
    "run_optuna": True,
    "n_epochs": 800,
    "batch_size": 256,
    "early_stop": 80,
    "optimizer": "Adam",
    "optim_hparas": {"lr": 0.0005, "weight_decay": 1e-5},
    "hidden_dims": [64, 32, 16],
    "dropout_rates": [0.3, 0.3, 0.2],
}

In [ ]:
# 5. Dataset
class COVID19Dataset(Dataset):
    """Dataset for loading and preprocessing the COVID19 dataset"""

    def __init__(self, path, mode="train", target_only=False, seed=42069):
        self.mode = mode
        self.seed = seed

        # Read data into numpy arrays
        with open(path, "r") as fp:
            data = list(csv.reader(fp))
            data = np.array(data[1:])[:, 1:].astype(float)

        if not target_only:
            self.feats = list(range(93))
        else:
            # Strong Baseline: 40 states + 2 tested_positive features
            self.feats = list(range(40)) + [57, 75]

        if mode == "test":
            # Testing data
            data = data[:, self.feats]
            self.data = torch.FloatTensor(data)
        else:
            # Training data
            target = data[:, -1]
            data = data[:, self.feats]

            # === 優化：改用 random_split ===
            dataset_size = len(data)
            indices = list(range(dataset_size))
            train_size = int(dataset_size * 0.9)
            dev_size = dataset_size - train_size

            g = torch.Generator().manual_seed(self.seed)
            train_indices, dev_indices = random_split(
                indices, [train_size, dev_size], generator=g
            )

            if mode == "train":
                selected_indices = train_indices
            else:  # dev
                selected_indices = dev_indices

            self.data = torch.FloatTensor(data[selected_indices])
            self.target = torch.FloatTensor(target[selected_indices])

        # === 優化：動態計算需標準化的特徵（非 state 特徵）===
        # state 特徵為前 40 維，其餘為 tested_positive 等需標準化
        self.normalize_start_idx = 40 if target_only else 0
        if self.data.shape[1] > self.normalize_start_idx:
            norm_data = self.data[:, self.normalize_start_idx :]
            self.data[:, self.normalize_start_idx :] = (
                norm_data - norm_data.mean(dim=0, keepdim=True)
            ) / norm_data.std(dim=0, keepdim=True)

            # # Splitting training data into train & dev sets
            # if mode == "train":
            #     indices = [i for i in range(len(data)) if i % 10 != 0]
            # elif mode == "dev":
            #     indices = [i for i in range(len(data)) if i % 10 == 0]

            # self.data = torch.FloatTensor(data[indices])
            # self.target = torch.FloatTensor(target[indices])

            # # Normalize features (only non-state features)
            # # After feature selection, the two tested_positive are at index 40, 41
            # if self.data.shape[1] > 40:
            #     self.data[:, 40:] = (
            #         self.data[:, 40:] - self.data[:, 40:].mean(dim=0, keepdim=True)
            #     ) / self.data[:, 40:].std(dim=0, keepdim=True)

        self.dim = self.data.shape[1]
        print(
            "Finished reading the {} set of COVID19 Dataset ({} samples found, each dim = {})".format(
                mode, len(self.data), self.dim
            )
        )

    def __getitem__(self, index):
        if self.mode in ["train", "dev"]:
            # For training
            return self.data[index], self.target[index]
        else:
            # For testing (no target)
            return self.data[index]

    def __len__(self):
        return len(self.data)

In [ ]:
# 6. DataLoader
def prep_dataloader(path, mode, batch_size, n_jobs=0, target_only=False):
    """Generates a dataset, then is put into a dataloader."""
    dataset = COVID19Dataset(path, mode=mode, target_only=target_only)
    dataloader = DataLoader(
        dataset,
        batch_size,
        shuffle=(mode == "train"),
        drop_last=False,
        num_workers=n_jobs,
        pin_memory=True,
    )
    return dataloader

In [ ]:
# 7. DNN
class NeuralNet(nn.Module):
    """A better fully-connected deep neural network"""

    def __init__(
        self, input_dim, hidden_dims=[64, 32, 16], dropout_rates=[0.3, 0.3, 0.2]
    ):
        super(NeuralNet, self).__init__()

        layers = []
        prev_dim = input_dim
        for i, h_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rates[i]))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))

        self.net = nn.Sequential(*layers)
        self.criterion = nn.MSELoss(reduction="mean")

    def forward(self, x):
        return self.net(x).squeeze(1)

    def cal_loss(self, pred, target):
        # L2 regularization is handled by optimizer (weight_decay)
        return self.criterion(pred, target)

In [ ]:
# 8. DNN Training
def train(tr_set, dv_set, model, config, device):
    n_epochs = config["n_epochs"]
    optimizer = getattr(torch.optim, config["optimizer"])(
        model.parameters(), **config["optim_hparas"]
    )

    min_mse = 1000.0
    loss_record = {"train": [], "dev": []}
    early_stop_cnt = 0
    epoch = 0

    while epoch < config["n_epochs"]:
        model.train()
        for x, y in tr_set:
            optimizer.zero_grad()
            x, y = x.to(device), y.to(device)
            pred = model(x)
            mse_loss = model.cal_loss(pred, y)
            mse_loss.backward()
            optimizer.step()
            loss_record["train"].append(mse_loss.detach().cpu().item())

        dev_mse = dev(dv_set, model, device)
        if dev_mse < min_mse:
            min_mse = dev_mse
            print(
                "Saving model (epoch = {:4d}, loss = {:.4f})".format(epoch + 1, min_mse)
            )
            torch.save(model.state_dict(), config["save_path"])
            early_stop_cnt = 0
        else:
            early_stop_cnt += 1

        epoch += 1
        loss_record["dev"].append(dev_mse)
        if early_stop_cnt > config["early_stop"]:
            break

    print("Finished training after {} epochs".format(epoch))
    return min_mse, loss_record

In [ ]:
# 9. Validation Loop
def dev(dv_set, model, device):
    model.eval()
    total_loss = 0
    for x, y in dv_set:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            pred = model(x)
            mse_loss = model.cal_loss(pred, y)
        total_loss += mse_loss.detach().cpu().item() * len(x)
    total_loss = total_loss / len(dv_set.dataset)

    return total_loss

In [ ]:
# 10. Inference Loop
def test(tt_set, model, device):
    model.eval()
    preds = []
    for x in tt_set:
        x = x.to(device)
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu())
    preds = torch.cat(preds, dim=0).numpy()
    return preds

In [ ]:
# 11. Load data and model
tr_set = prep_dataloader(
    tr_path, "train", config["batch_size"], target_only=config["target_only"]
)
dv_set = prep_dataloader(
    tr_path, "dev", config["batch_size"], target_only=config["target_only"]
)
tt_set = prep_dataloader(
    tt_path, "test", config["batch_size"], target_only=config["target_only"]
)

Finished reading the train set of COVID19 Dataset (2430 samples found, each dim = 42)
Finished reading the dev set of COVID19 Dataset (270 samples found, each dim = 42)
Finished reading the test set of COVID19 Dataset (893 samples found, each dim = 42)


In [ ]:
# 12. Optuna Hyperparameter Tuning
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold
import copy


def objective(trial, tr_set, dv_set, device):
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128])

    model = NeuralNet(
        tr_set.dataset.dim,
        hidden_dims=[hidden_dim, hidden_dim // 2, hidden_dim // 4],
        dropout_rates=[dropout_rate, dropout_rate, dropout_rate * 0.67],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 簡化訓練 loop
    n_epochs = 50  # 先用較少 epoch 快速過濾
    for epoch in range(n_epochs):
        model.train()
        for x, y in tr_set:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = model.cal_loss(pred, y)
            loss.backward()
            optimizer.step()

        # 驗證
        val_loss = dev(dv_set, model, device)
        trial.report(val_loss, epoch)

        # === Optuna 剪枝機制 ===
        if trial.should_prune():
            raise optuna.TrialPruned()

    return val_loss


def run_optuna_two_stage(tr_set, dv_set, device, n_trials=50, n_splits=5):
    # 第一階段：單一驗證集 + 剪枝，快速找 Top-3
    study = optuna.create_study(
        direction="minimize", pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    study.optimize(
        lambda trial: objective(trial, tr_set, dv_set, device), n_trials=n_trials
    )

    # 取得前 3 名參數
    top_trials = sorted(study.trials, key=lambda t: t.value)[:3]
    print("Top 3 hyperparameters:")
    for i, t in enumerate(top_trials):
        print(f"Rank {i + 1}: {t.params}, val_loss={t.value:.4f}")

    # 第二階段：只對 Top-3 跑 K 折
    best_kfold_models = []
    for rank, trial in enumerate(top_trials):
        print(f"\n=== Running K-Fold on Rank {rank + 1} params ===")
        params = trial.params
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

        fold_models = []
        for fold, (train_idx, val_idx) in enumerate(kf.split(tr_set.dataset)):
            model = NeuralNet(
                tr_set.dataset.dim,
                hidden_dims=[
                    params["hidden_dim"],
                    params["hidden_dim"] // 2,
                    params["hidden_dim"] // 4,
                ],
                dropout_rates=[
                    params["dropout_rate"],
                    params["dropout_rate"],
                    params["dropout_rate"] * 0.67,
                ],
            ).to(device)
            fold_models.append(model)
        best_kfold_models.append(fold_models)

    return best_kfold_models, top_trials

c:\Users\User\Documents\GitHub\auto-training-hub\Integrated_DL\1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 情況 A：若啟動自動調參
if config["run_optuna"]:
    import optuna

    est_models, top_params = run_optuna_two_stage(
        tr_set, dv_set, device, n_trials=30, n_splits=5
    )
    # 最終直接使用 best_fold_models 進行 K-Fold Average Blending 集成預測
    # save_pred(final_ensemble_preds, "pred_ensemble.csv")

# 情況 B：若不調參，直接用 config 跑單一模型實驗
else:
    model = NeuralNet(
        tr_set.dataset.dim, config["hidden_dims"], config["dropout_rates"]
    ).to(device)
    min_mse, loss_record = train(tr_set, dv_set, model, config, device)
    # 繪圖與單一模型輸出
    # preds = test(tt_set, model, device)
    # save_pred(preds, "pred.csv")

In [ ]:
# 13. K-Fold Ensemble Prediction
# 當不跑 Optuna 時，才需要手動跑這個單一模型與繪圖！
if not config["run_optuna"]:
    model_loss, model_loss_record = train(tr_set, dv_set, model, config, device)

    plot_learning_curve(model_loss_record, title="deep model")

    del model
    model = NeuralNet(tr_set.dataset.dim).to(device)
    ckpt = torch.load(config["save_path"], map_location="cpu")
    model.load_state_dict(ckpt)

    plot_pred(dv_set, model, device)

else:
    print("目前處於 Optuna 調參路線，已擁有 5 個 K-Fold 集成模型。")
    print("請直接跳過本區塊，前往 K-Fold 多模型集成預測」進行輸出。")

In [ ]:
# if not config["run_optuna"]:
#     preds = test(tt_set, model, device)  # predict COVID-19 cases with your model
#     save_pred(preds, "pred.csv")  # save prediction file to pred.csv

In [ ]:
# 14. K-Fold 多模型算術平均集成預測
if config["run_optuna"]:
    best_fold_models = est_models[0]

    # 將 5 個模型全部切換為評估模式
    for m in best_fold_models:
        m.eval()

    ensemble_preds = []

    # 遍歷測試集 DataLoader
    for x in tt_set:
        x = x.to(device)
        fold_preds = []

        with torch.no_grad():
            # 讓 5 個模型同時對這批資料進行預測
            for m in best_fold_models:
                fold_preds.append(m(x).cpu())

        # 將 5 個模型的預測值取算術平均 (Average Blending)
        batch_avg_pred = torch.stack(fold_preds, dim=0).mean(dim=0)
        ensemble_preds.append(batch_avg_pred)

    # 整合所有批次結果並轉為 NumPy
    final_ensemble_preds = torch.cat(ensemble_preds, dim=0).numpy()
    # 儲存集成預測結果
    save_pred(final_ensemble_preds, "pred_ensemble.csv")

Saving results to pred_ensemble.csv
